# Part II — Fine-tuning the ticket classifiers (Google Colab)
AI Service Desk Copilot · Group 60 · AIMLZG549 Assignment II

**Runtime → Change runtime type → T4 GPU** before you start. The whole notebook takes about 45–75 minutes.

What it produces (all needed for the report):
1. Clean train / validation / test splits and EDA charts
2. Two baselines: TF-IDF + logistic regression, and a zero-shot LLM
3. Fine-tuned DeBERTa-v3 **queue** classifier and **priority** classifier, logged to MLflow
4. Confusion matrices, training curves, before/after comparison tables
5. Models pushed to the Hugging Face Hub (used by the app) and a zip of all results

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Use **option A** after pushing the repo to GitHub, or **option B** to upload the zip.

In [ ]:
# Option A: clone from GitHub (edit the URL if your repo name differs)
REPO_URL = "https://github.com/prashob-narendran/servicedesk-copilot.git"  #@param {type:"string"}
!git clone -q $REPO_URL servicedesk-copilot

In [ ]:
# Option B: upload servicedesk-copilot.zip instead (skip if option A worked)
# from google.colab import files
# files.upload()
# !unzip -q servicedesk-copilot.zip

In [ ]:
%cd /content/servicedesk-copilot
!pip install -q -r requirements-train.txt

## 2. Secrets
Add these in Colab's **🔑 Secrets** panel (left sidebar) and enable notebook access:
- `OPENAI_API_KEY` — for the zero-shot baseline
- `HF_TOKEN` — a Hugging Face token with **write** access
- set `HF_USER` below to your Hugging Face username

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_USER = "your-hf-username"  #@param {type:"string"}
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## 3. Prepare the data
English tickets only, empty and duplicate tickets removed, 80/10/10 split stratified by queue (seed 42). If loading fails, the error lists the CSV files in the dataset repo; re-run with `--hub-file <name>`.

In [ ]:
!python -m data.prepare

In [ ]:
from IPython.display import Image, display
for f in ["queue_distribution", "priority_distribution", "ticket_length"]:
    display(Image(f"reports/figures/{f}.png", width=650))

## 4. Baselines (the 'before' numbers)

In [ ]:
!python -m classifier.baselines --label queue --zero-shot
!python -m classifier.baselines --label priority

## 5. Fine-tune the queue classifier
About 20–35 minutes on a T4. If you run out of time, add `--model distilroberta-base` (about 3× faster, slightly lower accuracy).

In [ ]:
!python -m classifier.train --label queue --push-to-hub {HF_USER}/servicedesk-queue-classifier

## 6. Fine-tune the priority classifier

In [ ]:
!python -m classifier.train --label priority --push-to-hub {HF_USER}/servicedesk-priority-classifier

## 7. Results for the report

In [ ]:
!python -m classifier.compare --label queue
!python -m classifier.compare --label priority

In [ ]:
from IPython.display import Markdown
for lab in ["queue", "priority"]:
    display(Markdown(open(f"reports/metrics/comparison_{lab}.md").read()))
    display(Image(f"reports/figures/confusion_matrix_{lab}.png", width=650))
    display(Image(f"reports/figures/training_curves_{lab}.png", width=650))

In [ ]:
import json
for lab in ["queue", "priority"]:
    m = json.load(open(f"reports/metrics/finetuned_{lab}.json"))
    print(f"\n== {lab}: most confused pairs ==")
    for c in m["top_confusions"]:
        print(f"{c['true']} -> {c['predicted']}: {c['count']} tickets")
        for ex in c["examples"][:2]:
            print("   •", ex[:160])

## 8. MLflow runs (screenshot this table for the report)

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["servicedesk-copilot"])
cols = [c for c in runs.columns if c.startswith(("params.label", "params.base_model", "metrics.test_", "metrics.eval_macro_f1", "metrics.train_minutes"))]
runs[["run_id", "start_time"] + cols]

## 9. Download everything
The models are on the Hub already. This zip holds the metrics, charts and MLflow database for the report, plus the past tickets for the RAG index.

In [ ]:
!zip -qr part2_results.zip reports mlflow.db data/processed/kb_tickets.csv models/*/README.md models/*/metrics.json models/*/label_map.json
from google.colab import files
files.download("part2_results.zip")

Next, on your laptop, set in `.env`:
```
CLASSIFIER_MODEL_ID=<HF_USER>/servicedesk-queue-classifier
PRIORITY_MODEL_ID=<HF_USER>/servicedesk-priority-classifier
```
unzip it into the repo folder, then rebuild the RAG index with the past tickets: `python -m rag.ingest --tickets data/processed/kb_tickets.csv --reset`.